# Clipping analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
mimicdata = pd.read_parquet("../Dataset/mimic-iv/from_pipeline/ltm_grided.parquet")

In [3]:
L_to_clip = [
 'bicarbonate_arterial__last__last_12h',
 'activated_partial_thromboplastin_time__last__last_12h',
 'temperature__mean__last_12h',
 'hemoglobin__last__last_12h',
 'heart_rate__mean__last_12h',
 'arterial_blood_pressure_mean__mean__last_12h',
 'creatinine__last__last_12h',
 'ureum__last__last_12h',
 'fluid_out_urine__mean__last_12h',
 'lactate__last__last_12h',
 #'glasgow_coma_scale_total__last__last_12h',
 'pco2_arterial__mean__last_12h',
 'respiratory_rate_measured__mean__last_12h',
 'o2_saturation__mean__last_12h',
 'fio2__last__last_12h',
 'po2_arterial__mean__last_12h',
 #'vent_mode__last__last_12h',
 #'vent_mode__hours_since_last__last_12h'
 ]

In [4]:
# -----------------------------
# CONFIG
# -----------------------------
# Example:
# mimicdata = ...
# L_to_clip = [...]

ID_COL_CANDIDATES = ["stay_id", "hadm_id", "subject_id"]
TIME_COL_CANDIDATES = ["t0", "time", "grid_end"]

# Optional: manually fill units if you already know them
UNIT_MAP = {
    "bicarbonate_arterial__last__last_12h": None,
    "activated_partial_thromboplastin_time__last__last_12h": None,
    "temperature__mean__last_12h": None,
    "hemoglobin__last__last_12h": None,
    "heart_rate__mean__last_12h": None,
    "arterial_blood_pressure_mean__mean__last_12h": None,
    "creatinine__last__last_12h": None,
    "ureum__last__last_12h": None,
    "fluid_out_urine__mean__last_12h": None,
    "lactate__last__last_12h": None,
    "pco2_arterial__mean__last_12h": None,
    "respiratory_rate_measured__mean__last_12h": None,
    "o2_saturation__mean__last_12h": None,
    "fio2__last__last_12h": None,
    "po2_arterial__mean__last_12h": None,
}

# Optional: define provenance if you know it
SOURCE_MAP = {
    # "variable_name": {"source_table": "chartevents/labevents", "itemids_used": "...", "aggregation_definition": "..."}
}

# -----------------------------
# HELPERS
# -----------------------------
def pick_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def safe_mode(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    m = s.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan

def rounded_value_counts(series, decimals=2, top_n=20):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return []
    vc = s.round(decimals).value_counts(dropna=False).head(top_n)
    return [(float(idx), int(cnt)) for idx, cnt in vc.items()]

def exact_value_counts(series, top_n=20):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return []
    vc = s.value_counts(dropna=False).head(top_n)
    out = []
    for idx, cnt in vc.items():
        try:
            idx_val = float(idx)
        except Exception:
            idx_val = str(idx)
        out.append((idx_val, int(cnt)))
    return out

def summarize_missingness_over_time(df, var, time_col, n_bins=5):
    if time_col is None or time_col not in df.columns:
        return {
            "pct_missing_early": np.nan,
            "pct_missing_mid": np.nan,
            "pct_missing_late": np.nan
        }

    tmp = df[[time_col, var]].copy()
    tmp = tmp.dropna(subset=[time_col])

    if tmp.empty:
        return {
            "pct_missing_early": np.nan,
            "pct_missing_mid": np.nan,
            "pct_missing_late": np.nan
        }

    # convert timedeltas if needed
    t = tmp[time_col]
    if pd.api.types.is_timedelta64_dtype(t):
        t_num = t.dt.total_seconds() / 3600.0
    else:
        t_num = pd.to_numeric(t, errors="coerce")

    tmp = tmp.assign(_time_num=t_num).dropna(subset=["_time_num"])
    if tmp.empty:
        return {
            "pct_missing_early": np.nan,
            "pct_missing_mid": np.nan,
            "pct_missing_late": np.nan
        }

    # quantile-based bins
    try:
        tmp["_bin"] = pd.qcut(tmp["_time_num"], q=n_bins, duplicates="drop")
    except ValueError:
        return {
            "pct_missing_early": np.nan,
            "pct_missing_mid": np.nan,
            "pct_missing_late": np.nan
        }

    miss_by_bin = tmp.groupby("_bin", observed=False)[var].apply(lambda x: x.isna().mean() * 100)

    if len(miss_by_bin) == 0:
        return {
            "pct_missing_early": np.nan,
            "pct_missing_mid": np.nan,
            "pct_missing_late": np.nan
        }

    vals = miss_by_bin.tolist()
    early = vals[0]
    mid = vals[len(vals)//2]
    late = vals[-1]

    return {
        "pct_missing_early": early,
        "pct_missing_mid": mid,
        "pct_missing_late": late
    }

def maybe_convert_time_to_numeric(series):
    if pd.api.types.is_timedelta64_dtype(series):
        return series.dt.total_seconds() / 3600.0
    return pd.to_numeric(series, errors="coerce")

# -----------------------------
# MAIN SUMMARY
# -----------------------------
def build_clipping_summary(
    mimicdata,
    L_to_clip,
    unit_map=None,
    source_map=None,
    top_n_values=20,
    rounded_decimals=2
):
    unit_map = unit_map or {}
    source_map = source_map or {}

    df = mimicdata.copy()
    time_col = pick_first_existing(df.columns, TIME_COL_CANDIDATES)
    id_col = pick_first_existing(df.columns, ID_COL_CANDIDATES)

    summary_rows = []
    top_values_rows = []

    for var in L_to_clip:
        if var not in df.columns:
            summary_rows.append({
                "variable": var,
                "present_in_dataframe": False
            })
            continue

        s_raw = df[var]
        s = pd.to_numeric(s_raw, errors="coerce")

        n_total = len(df)
        n_nonmissing = s.notna().sum()
        pct_missing = 100 * (1 - n_nonmissing / n_total) if n_total > 0 else np.nan

        if n_nonmissing == 0:
            summary_rows.append({
                "variable": var,
                "present_in_dataframe": True,
                "unit": unit_map.get(var),
                "n_total_rows": n_total,
                "n_nonmissing": 0,
                "pct_missing": pct_missing,
                "n_unique_nonmissing": 0,
                "min": np.nan,
                "p001": np.nan,
                "p01": np.nan,
                "p05": np.nan,
                "median": np.nan,
                "mean": np.nan,
                "mode": np.nan,
                "p95": np.nan,
                "p99": np.nan,
                "p999": np.nan,
                "max": np.nan,
                "iqr": np.nan,
                "std": np.nan,
                "n_zero": np.nan,
                "pct_zero_among_nonmissing": np.nan,
                "n_negative": np.nan,
                "pct_negative_among_nonmissing": np.nan,
                "n_lt_0": np.nan,
                "n_gt_0": np.nan,
                "pct_missing_early": np.nan,
                "pct_missing_mid": np.nan,
                "pct_missing_late": np.nan,
                "source_table": source_map.get(var, {}).get("source_table"),
                "itemids_used": source_map.get(var, {}).get("itemids_used"),
                "aggregation_definition": source_map.get(var, {}).get("aggregation_definition"),
            })
            continue

        q = s.quantile([0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999])
        q001 = q.loc[0.001]
        q01 = q.loc[0.01]
        q05 = q.loc[0.05]
        q25 = q.loc[0.25]
        q50 = q.loc[0.5]
        q75 = q.loc[0.75]
        q95 = q.loc[0.95]
        q99 = q.loc[0.99]
        q999 = q.loc[0.999]

        iqr = q75 - q25
        mode_val = safe_mode(s)

        miss_time = summarize_missingness_over_time(df, var, time_col)

        summary_rows.append({
            "variable": var,
            "present_in_dataframe": True,
            "unit": unit_map.get(var),
            "n_total_rows": n_total,
            "n_nonmissing": int(n_nonmissing),
            "pct_missing": pct_missing,
            "n_unique_nonmissing": int(s.nunique(dropna=True)),
            "min": s.min(),
            "p001": q001,
            "p01": q01,
            "p05": q05,
            "median": q50,
            "mean": s.mean(),
            "mode": mode_val,
            "p95": q95,
            "p99": q99,
            "p999": q999,
            "max": s.max(),
            "iqr": iqr,
            "std": s.std(),
            "n_zero": int((s == 0).sum()),
            "pct_zero_among_nonmissing": 100 * ((s == 0).sum() / n_nonmissing),
            "n_negative": int((s < 0).sum()),
            "pct_negative_among_nonmissing": 100 * ((s < 0).sum() / n_nonmissing),
            "n_lt_0": int((s < 0).sum()),
            "n_gt_0": int((s > 0).sum()),
            "pct_missing_early": miss_time["pct_missing_early"],
            "pct_missing_mid": miss_time["pct_missing_mid"],
            "pct_missing_late": miss_time["pct_missing_late"],
            "source_table": source_map.get(var, {}).get("source_table"),
            "itemids_used": source_map.get(var, {}).get("itemids_used"),
            "aggregation_definition": source_map.get(var, {}).get("aggregation_definition"),
        })

        exact_top = exact_value_counts(s, top_n=top_n_values)
        rounded_top = rounded_value_counts(s, decimals=rounded_decimals, top_n=top_n_values)

        for rank, (val, cnt) in enumerate(exact_top, start=1):
            top_values_rows.append({
                "variable": var,
                "value_type": "exact",
                "rank": rank,
                "value": val,
                "count": cnt
            })

        for rank, (val, cnt) in enumerate(rounded_top, start=1):
            top_values_rows.append({
                "variable": var,
                "value_type": f"rounded_{rounded_decimals}dp",
                "rank": rank,
                "value": val,
                "count": cnt
            })

    summary_df = pd.DataFrame(summary_rows)
    top_values_df = pd.DataFrame(top_values_rows)

    return summary_df, top_values_df, time_col, id_col

# -----------------------------
# OPTIONAL STAY-LEVEL SUMMARY
# useful if repeated measures inflate tails
# -----------------------------
def build_stay_level_extremes(mimicdata, L_to_clip, id_col=None):
    if id_col is None or id_col not in mimicdata.columns:
        return pd.DataFrame()

    rows = []
    for var in L_to_clip:
        if var not in mimicdata.columns:
            continue

        s = pd.to_numeric(mimicdata[var], errors="coerce")
        tmp = pd.DataFrame({id_col: mimicdata[id_col], var: s}).dropna(subset=[var])

        if tmp.empty:
            continue

        per_stay_max = tmp.groupby(id_col)[var].max()
        per_stay_min = tmp.groupby(id_col)[var].min()

        rows.append({
            "variable": var,
            "stay_n": per_stay_max.shape[0],
            "stay_level_min_of_mins": per_stay_min.min(),
            "stay_level_p01_of_mins": per_stay_min.quantile(0.01),
            "stay_level_p50_of_mins": per_stay_min.quantile(0.50),
            "stay_level_p99_of_mins": per_stay_min.quantile(0.99),
            "stay_level_max_of_mins": per_stay_min.max(),
            "stay_level_min_of_maxs": per_stay_max.min(),
            "stay_level_p01_of_maxs": per_stay_max.quantile(0.01),
            "stay_level_p50_of_maxs": per_stay_max.quantile(0.50),
            "stay_level_p99_of_maxs": per_stay_max.quantile(0.99),
            "stay_level_max_of_maxs": per_stay_max.max(),
        })

    return pd.DataFrame(rows)

# -----------------------------
# OPTIONAL PLOTS
# -----------------------------
def make_clipping_plots_pdf(mimicdata, L_to_clip, pdf_path="clipping_plots.pdf"):
    with PdfPages(pdf_path) as pdf:
        for var in L_to_clip:
            if var not in mimicdata.columns:
                continue

            s = pd.to_numeric(mimicdata[var], errors="coerce").dropna()
            if s.empty:
                continue

            fig = plt.figure(figsize=(11, 8))

            # Histogram raw
            ax1 = fig.add_subplot(2, 2, 1)
            ax1.hist(s, bins=60)
            ax1.set_title(f"{var} - histogram (raw)")
            ax1.set_xlabel("value")
            ax1.set_ylabel("count")

            # Boxplot raw
            ax2 = fig.add_subplot(2, 2, 2)
            ax2.boxplot(s.values, vert=True)
            ax2.set_title(f"{var} - boxplot (raw)")
            ax2.set_ylabel("value")

            # Histogram log1p only if nonnegative
            ax3 = fig.add_subplot(2, 2, 3)
            if (s >= 0).all():
                ax3.hist(np.log1p(s), bins=60)
                ax3.set_title(f"{var} - histogram log1p")
                ax3.set_xlabel("log1p(value)")
            else:
                ax3.hist(s.clip(lower=s.quantile(0.01), upper=s.quantile(0.99)), bins=60)
                ax3.set_title(f"{var} - histogram clipped to p01-p99")
                ax3.set_xlabel("value")

            # Tail-focused histogram
            ax4 = fig.add_subplot(2, 2, 4)
            lo, hi = s.quantile(0.01), s.quantile(0.99)
            s_tail = s[(s >= lo) & (s <= hi)]
            ax4.hist(s_tail, bins=60)
            ax4.set_title(f"{var} - histogram p01 to p99")
            ax4.set_xlabel("value")
            ax4.set_ylabel("count")

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

# -----------------------------
# RUN
# -----------------------------
summary_df, top_values_df, time_col, id_col = build_clipping_summary(
    mimicdata=mimicdata,
    L_to_clip=L_to_clip,
    unit_map=UNIT_MAP,
    source_map=SOURCE_MAP,
    top_n_values=20,
    rounded_decimals=2
)

stay_level_df = build_stay_level_extremes(
    mimicdata=mimicdata,
    L_to_clip=L_to_clip,
    id_col=id_col
)

make_clipping_plots_pdf(
    mimicdata=mimicdata,
    L_to_clip=L_to_clip,
    pdf_path="clipping_plots.pdf"
)

summary_df.to_csv("clipping_summary.csv", index=False)
top_values_df.to_csv("clipping_top_values.csv", index=False)
stay_level_df.to_csv("clipping_stay_level_extremes.csv", index=False)

print("Detected time column:", time_col)
print("Detected ID column:", id_col)
print("Saved:")
print(" - clipping_summary.csv")
print(" - clipping_top_values.csv")
print(" - clipping_stay_level_extremes.csv")
print(" - clipping_plots.pdf")

# Show preview
display(summary_df)
display(top_values_df.head(50))
display(stay_level_df)

Detected time column: grid_end
Detected ID column: stay_id
Saved:
 - clipping_summary.csv
 - clipping_top_values.csv
 - clipping_stay_level_extremes.csv
 - clipping_plots.pdf


,variable,present_in_dataframe,unit,n_total_rows,n_nonmissing,pct_missing,n_unique_nonmissing,min,p001,p01,...,n_negative,pct_negative_among_nonmissing,n_lt_0,n_gt_0,pct_missing_early,pct_missing_mid,pct_missing_late,source_table,itemids_used,aggregation_definition
0,bicarbonate_arterial__last__last_12h,True,None,718198,17317,97.588826,46,5.000000,8.000000,13.000000,...,0,0.000000,0,17317,97.310582,97.769988,97.546047,None,None,None
1,activated_partial_thromboplastin_time__last__l...,True,None,718198,11308,98.425504,1002,18.000000,19.230700,21.700001,...,0,0.000000,0,11308,98.228418,98.553927,98.390644,None,None,None
2,temperature__mean__last_12h,True,None,718198,616207,14.200959,7127,-17.777779,32.480401,35.666668,...,18,0.002921,18,616189,10.186742,17.701296,9.946075,None,None,None
3,hemoglobin__last__last_12h,True,None,718198,17356,97.583396,145,2.800000,5.300000,6.400000,...,0,0.000000,0,17356,97.057751,97.740245,97.770152,None,None,None
4,heart_rate__mean__last_12h,True,None,718198,677969,5.601380,21443,-20027.083984,43.076290,52.615383,...,1,0.000147,1,677949,0.208733,9.002903,3.487639,None,None,None
5,arterial_blood_pressure_mean__mean__last_12h,True,None,718198,661140,7.944606,34122,0.000000,42.500000,55.583332,...,0,0.000000,0,661132,1.248868,11.552298,6.793893,None,None,None
6,creatinine__last__last_12h,True,None,718198,17344,97.585067,140,0.000000,8.840000,26.520002,...,0,0.000000,0,17338,97.281183,97.754408,97.575460,None,None,None
7,ureum__last__last_12h,True,None,718198,17329,97.587156,182,0.357000,0.714000,1.428000,...,0,0.000000,0,17329,97.291179,97.765031,97.565656,None,None,None
8,fluid_out_urine__mean__last_12h,True,None,718198,437340,39.105929,13069,-287.083344,0.000000,2.857143,...,2,0.000457,2,435032,34.005786,40.776149,42.767701,None,None,None
9,lactate__last__last_12h,True,None,718198,9562,98.668612,190,0.300000,0.500000,0.600000,...,0,0.000000,0,9562,98.041441,98.898095,98.870369,None,None,None


,variable,value_type,rank,value,count
0,bicarbonate_arterial__last__last_12h,exact,1,24.000000,1639
1,bicarbonate_arterial__last__last_12h,exact,2,25.000000,1503
2,bicarbonate_arterial__last__last_12h,exact,3,23.000000,1473
3,bicarbonate_arterial__last__last_12h,exact,4,26.000000,1396
4,bicarbonate_arterial__last__last_12h,exact,5,22.000000,1338
5,bicarbonate_arterial__last__last_12h,exact,6,27.000000,1211
6,bicarbonate_arterial__last__last_12h,exact,7,21.000000,1125
7,bicarbonate_arterial__last__last_12h,exact,8,28.000000,972
8,bicarbonate_arterial__last__last_12h,exact,9,20.000000,921
9,bicarbonate_arterial__last__last_12h,exact,10,29.000000,766


,variable,stay_n,stay_level_min_of_mins,stay_level_p01_of_mins,stay_level_p50_of_mins,stay_level_p99_of_mins,stay_level_max_of_mins,stay_level_min_of_maxs,stay_level_p01_of_maxs,stay_level_p50_of_maxs,stay_level_p99_of_maxs,stay_level_max_of_maxs
0,bicarbonate_arterial__last__last_12h,11515,5.000000,12.000000,24.000000,39.000000,50.000000,5.000000,13.000000,25.000000,40.000000,5.000000e+01
1,activated_partial_thromboplastin_time__last__l...,8083,18.000000,21.299999,33.400002,150.000000,150.000000,18.299999,21.582000,35.599998,150.000000,1.500000e+02
2,temperature__mean__last_12h,83337,-17.777779,34.678780,36.574078,37.486115,40.750000,-17.777779,36.074075,37.074078,39.013892,4.337476e+04
3,hemoglobin__last__last_12h,11805,2.800000,6.200000,9.300000,15.100000,999999.000000,3.400000,6.500000,9.700000,15.400000,9.999990e+05
4,heart_rate__mean__last_12h,85033,-20027.083984,46.098740,74.368423,112.931811,156.000000,30.000000,58.750000,92.583336,139.000000,4.261525e+05
5,arterial_blood_pressure_mean__mean__last_12h,84917,0.000000,46.784540,71.076920,102.666664,599.166687,0.000000,62.153847,86.555557,127.152089,2.963025e+06
6,creatinine__last__last_12h,11579,0.000000,26.520002,88.400002,716.040039,2793.440186,8.840000,26.520002,88.400002,760.240051,2.793440e+03
7,ureum__last__last_12h,11563,0.357000,1.428000,7.497000,41.190660,76.397995,0.357000,1.428000,8.210999,45.695999,7.639799e+01
8,fluid_out_urine__mean__last_12h,61616,-287.083344,0.000000,56.000000,550.000000,3100.000000,0.000000,7.262500,185.833328,1043.187500,3.533333e+03
9,lactate__last__last_12h,6519,0.300000,0.600000,1.500000,13.800000,26.000000,0.300000,0.600000,1.800000,15.882000,2.600000e+01
